# Module 2 Homework (2026 Cohort) — Solutions

Notebook ini mengerjakan Q1–Q5 dari `homework2.md`.

In [1]:
import io
import os
import re
import time
import warnings

import numpy as np
import pandas as pd
import requests
import yfinance as yf

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0 Safari/537.36'
    )
}

def read_html_table(url: str) -> pd.DataFrame:
    """Download a page and return its first HTML table (wrapped in StringIO for Windows/new pandas)."""
    resp = requests.get(url, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return pd.read_html(io.StringIO(resp.text))[0]

---
## Question 1: [IPO] Withdrawn IPOs by Company Type

### 1. Data loading

In [2]:
ipos_filed = read_html_table('https://www.iposcoop.com/ipos-recently-filed/')
ipos_filed['File Date'] = pd.to_datetime(ipos_filed['File Date'])
print(ipos_filed.shape)
ipos_filed.head()

(500, 10)


,File Date,Company,Symbol,Managers,Shares (millions),Price Low,Price High,Est $ Vol (millions),Expected To Trade,SCOOP Rating
0,2026-09-22,Accelevation Holdings Corp.,ACCV,Morgan Stanley/J.P. Morgan/Goldman Sachs/Barcl...,30.00,$20.00,$24.00,$660.00,Wednesday,S/O
1,2026-09-22,Live Oak Acquisition VI,LOVIU,Santander,20.00,$10.00,$10.00,$200.00,Priced,S/O
2,2026-09-21,ADARx Pharmaceuticals,ADRX,J.P.Morgan/Morgan Stanley/TD Cowen/UBS Investm...,21.88,$15.00,$17.00,$350.00,Friday,S/O
3,2026-09-21,Iambic Therapeutics,IAM,J.P.Morgan/Jefferies/BofA Securities/Citigroup,0.00,NaN,NaN,$100.00,TBA,S/O
4,2026-09-21,Oura,OURA,Goldman Sachs/Morgan Stanley/J.P.Morgan/Allen ...,50.00,$40.00,$44.00,$2100.00,Wednesday,S/O


In [3]:
# keep only withdrawn IPOs filed before Sep 11, 2026
withdrawn = ipos_filed[
    (ipos_filed['Expected To Trade'] == 'Withdrawn')
    & (ipos_filed['File Date'] < '2026-09-11')
].copy()
print('Withdrawn entries:', len(withdrawn))
withdrawn[['File Date', 'Company', 'Symbol', 'Shares (millions)', 'Price Low', 'Price High', 'Est $ Vol (millions)']]

Withdrawn entries: 31


,File Date,Company,Symbol,Shares (millions),Price Low,Price High,Est $ Vol (millions)
30,2026-09-10,Motive Technologies (Withdrawn),MTVE,0.00,NaN,NaN,$100.00
36,2026-09-04,Idea Tech Holding (Withdrawn),IDTL,2.00,$4.00,$5.00,$9.00
44,2026-09-01,Hornbeck Offshore Services (Withdrawn),HOS,0.00,NaN,NaN,$100.00
46,2026-08-31,Coolbit Technologies Ltd. (Withdrawn),CBAI,5.00,$4.00,$5.00,$22.50
54,2026-08-27,Timber Road Acquisition Corp. (Withdrawn),TMRDU,20.00,$10.00,$10.00,$200.00
64,2026-08-24,pan-Africa Corp. (Withdrawn),BLEUU,12.50,$10.00,$10.00,$125.00
79,2026-08-18,Living Homeopathy (Withdrawn),LHI,3.75,$4.00,$6.00,$19.00
122,2026-07-31,Nuclea Energy (Withdrawn),NCLA,5.60,$8.00,$10.00,$50.40
152,2026-07-10,APEX Global Solutions Ltd. (WITHDRAWN),APEX,3.75,$4.00,$4.00,$15.00
157,2026-07-08,Cloud Data Holdings (Withdrawn),CDN,3.75,$4.00,$4.50,$15.94


> Catatan: halaman iposcoop hanya menampilkan 500 filing terakhir, sehingga jumlah baris bisa sedikit berbeda dari 32 di soal tergantung kapan data diambil.

### 2. Company classification

In [4]:
def classify_company(name: str) -> str:
    # order matters: first matched rule wins
    rules = [
        (['Technologies'], 'Technologies'),
        (['Acquisition Corp', 'Acquisition Corporation', 'Corp'], 'Acquisition Corp'),
        (['Inc', 'Incorporated'], 'Inc.'),
        (['Group'], 'Group'),
        (['Ltd', 'Limited'], 'Limited'),
        (['Holdings', 'Holding'], 'Holdings'),
    ]
    for patterns, label in rules:
        if any(p in name for p in patterns):
            return label
    return 'Other'

withdrawn['Company Type'] = withdrawn['Company'].apply(classify_company)
withdrawn[['Company', 'Company Type']]

,Company,Company Type
30,Motive Technologies (Withdrawn),Technologies
36,Idea Tech Holding (Withdrawn),Holdings
44,Hornbeck Offshore Services (Withdrawn),Other
46,Coolbit Technologies Ltd. (Withdrawn),Technologies
54,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp
64,pan-Africa Corp. (Withdrawn),Acquisition Corp
79,Living Homeopathy (Withdrawn),Other
122,Nuclea Energy (Withdrawn),Other
152,APEX Global Solutions Ltd. (WITHDRAWN),Limited
157,Cloud Data Holdings (Withdrawn),Holdings


### 3–5. Price parsing, numeric conversion, value calculation

In [5]:
def parse_price(value):
    """'$8.00' -> 8.0 ; '-' or missing -> NaN"""
    if pd.isna(value):
        return np.nan
    s = str(value).replace('$', '').replace(',', '').strip()
    if s in ('', '-'):
        return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

def to_number(series: pd.Series) -> pd.Series:
    cleaned = series.astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip()
    return pd.to_numeric(cleaned, errors='coerce')

withdrawn['Avg_price'] = (withdrawn['Price Low'].apply(parse_price) + withdrawn['Price High'].apply(parse_price)) / 2
withdrawn['Shares (millions)'] = to_number(withdrawn['Shares (millions)'])
withdrawn['Est $ Vol (millions)'] = to_number(withdrawn['Est $ Vol (millions)'])

shares_x_price = withdrawn['Shares (millions)'] * withdrawn['Avg_price']
withdrawn['Shares_offered_value'] = np.where(shares_x_price.notna(), shares_x_price, withdrawn['Est $ Vol (millions)'])

withdrawn[['Company', 'Company Type', 'Shares (millions)', 'Avg_price', 'Est $ Vol (millions)', 'Shares_offered_value']]

,Company,Company Type,Shares (millions),Avg_price,Est $ Vol (millions),Shares_offered_value
30,Motive Technologies (Withdrawn),Technologies,0.00,NaN,100.00,100.0000
36,Idea Tech Holding (Withdrawn),Holdings,2.00,4.50,9.00,9.0000
44,Hornbeck Offshore Services (Withdrawn),Other,0.00,NaN,100.00,100.0000
46,Coolbit Technologies Ltd. (Withdrawn),Technologies,5.00,4.50,22.50,22.5000
54,Timber Road Acquisition Corp. (Withdrawn),Acquisition Corp,20.00,10.00,200.00,200.0000
64,pan-Africa Corp. (Withdrawn),Acquisition Corp,12.50,10.00,125.00,125.0000
79,Living Homeopathy (Withdrawn),Other,3.75,5.00,19.00,18.7500
122,Nuclea Energy (Withdrawn),Other,5.60,9.00,50.40,50.4000
152,APEX Global Solutions Ltd. (WITHDRAWN),Limited,3.75,4.00,15.00,15.0000
157,Cloud Data Holdings (Withdrawn),Holdings,3.75,4.25,15.94,15.9375


### 6. Aggregation

In [6]:
q1 = withdrawn.groupby('Company Type')['Shares_offered_value'].sum().sort_values(ascending=False)
print(q1)
print(f"\nANSWER Q1: {q1.index[0]} with total withdrawn value ${q1.iloc[0]:,.2f} million")

Company Type
Acquisition Corp    499.9850
Inc.                351.0000
Holdings            311.6575
Other               290.4450
Limited             197.2500
Technologies        184.9000
Group                32.5000
Name: Shares_offered_value, dtype: float64

ANSWER Q1: Acquisition Corp with total withdrawn value $499.99 million


---
## Question 2: [IPO] Median Sharpe Ratio for 2025 IPOs (First 8 Months)

In [7]:
ipos_2025 = read_html_table('https://www.iposcoop.com/2025-pricings/')
ipos_2025['Offer Date'] = pd.to_datetime(ipos_2025['Offer Date'], format='%m/%d/%Y')
print('All 2025 IPOs:', len(ipos_2025))

ipos_filtered = ipos_2025[
    (ipos_2025['Offer Date'] < '2025-09-01')
    & (ipos_2025['Return'] != '0.00%')
].copy()
print('Filtered IPOs (before 2025-09-01, non-zero return):', len(ipos_filtered))
ipos_filtered.head()

All 2025 IPOs: 231
Filtered IPOs (before 2025-09-01, non-zero return): 146


,Company,Symbol,Industry,Offer Date,Shares (millions),Offer Price,1st Day Close,Current Price,Return,SCOOP Rating
68,"Picard Medical, Inc.",PMI,Health Care,2025-08-29,4.3,$4.00,$4.59,$1.68,-58.00%,S/O
69,GrowHub Limited (The),TGHL,Technology,2025-08-28,3.7,$4.00,$3.38,$0.40,-90.00%,S/O
70,TryHard Holdings Limited,THH,Consumer Services,2025-08-28,1.5,$4.00,$6.02,$29.47,636.75%,S/O
71,Curanex Pharmaceuticals,CURX,Health Care,2025-08-26,3.8,$4.00,$4.02,$0.37,-90.75%,S/O
72,"Cantor Equity Partners IV, Inc. (Stock-Only SPAC)",CEPF,Blank Check,2025-08-21,40.0,$10.00,$10.12,$10.12,1.20%,S/O


In [8]:
TICKERS = ipos_filtered['Symbol'].str.strip().unique().tolist()

frames, missing = [], []
for ticker in TICKERS:
    try:
        hist = yf.download(ticker, period='max', interval='1d', progress=False, auto_adjust=True)
    except Exception:
        hist = pd.DataFrame()
    if hist is None or hist.empty:
        missing.append(ticker)  # probably delisted / not on Yahoo Finance
        continue
    # flatten MultiIndex columns returned by recent yfinance versions
    if isinstance(hist.columns, pd.MultiIndex):
        hist.columns = hist.columns.get_level_values(0)
    hist = hist.dropna(subset=['Close'])
    if hist.empty:
        missing.append(ticker)
        continue
    hist['Ticker'] = ticker
    hist['Date'] = hist.index.tz_localize(None) if hist.index.tz is not None else hist.index
    frames.append(hist.reset_index(drop=True))
    time.sleep(0.2)

stocks_df = pd.concat(frames, ignore_index=True)
print('Tickers downloaded:', stocks_df['Ticker'].nunique())
print('Missing / delisted:', len(missing), missing)

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: MJID"}}}
$MJID: possibly delisted; no timezone found

1 Failed download:
['MJID']: possibly delisted; no timezone found
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: EMPG"}}}
$EMPG: possibly delisted; no timezone found

1 Failed download:
['EMPG']: possibly delisted; no timezone found
$CAEP: possibly delisted; no timezone found

1 Failed download:
['CAEP']: possibly delisted; no timezone found
$PTNM: possibly delisted; no timezone found

1 Failed download:
['PTNM']: possibly delisted; no timezone found
$AHL: possibly delisted; no timezone found

1 Failed download:
['AHL']: possibly delisted; no timezone found
$CEPT: possibly delisted; no timezone found

1 Failed download:
['CEPT']: possibly delisted; no timezone found
$SDM: possibly delisted; no timezone found

1 Failed download:
['SDM']: possibly d

Tickers downloaded: 133
Missing / delisted: 13 ['MJID', 'EMPG', 'CAEP', 'PTNM', 'AHL', 'CEPT', 'SDM', 'TBH', 'AGH', 'EPWK', 'MTSR', 'SKBL', 'MCTR']


In [9]:
stocks_df = stocks_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
g = stocks_df.groupby('Ticker')['Close']

stocks_df['growth_252d'] = stocks_df['Close'] / g.shift(252)
stocks_df['volatility'] = g.transform(lambda s: s.rolling(30).std()) * np.sqrt(252)
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.05) / stocks_df['volatility']

q2 = stocks_df[stocks_df['Date'] == '2026-09-11']
q2[['growth_252d', 'volatility', 'Sharpe']].describe()

Price,growth_252d,volatility,Sharpe
count,130.000000,131.000000,130.000000
mean,1.057613,27.488300,NaN
std,3.049007,66.890026,NaN
min,0.001005,0.000000,-inf
25%,0.141035,2.265163,0.012042
50%,0.594534,7.885748,0.050142
75%,1.038821,23.452319,0.133619
max,33.638270,627.698106,inf


In [10]:
print('Stocks with a 252-day history on 2026-09-11:', q2['growth_252d'].notna().sum())
print(f"growth_252d  mean={q2['growth_252d'].mean():.4f}  median={q2['growth_252d'].median():.4f}")
print(f"\nANSWER Q2: median Sharpe ratio on 2026-09-11 = {q2['Sharpe'].median():.4f}")

Stocks with a 252-day history on 2026-09-11: 130
growth_252d  mean=1.0576  median=0.5945

ANSWER Q2: median Sharpe ratio on 2026-09-11 = 0.0501


In [11]:
# Risk-adjusted view: compare Sharpe ranking vs pure growth ranking
tmp = q2.dropna(subset=['Sharpe'])[['Ticker', 'Close', 'growth_252d', 'volatility', 'Sharpe']].copy()
tmp['growth_rank'] = tmp['growth_252d'].rank(ascending=False)
tmp['sharpe_rank'] = tmp['Sharpe'].rank(ascending=False)
tmp.sort_values('Sharpe', ascending=False).head(10)

Price,Ticker,Close,growth_252d,volatility,Sharpe,growth_rank,sharpe_rank
44604,MAMK,13.000,6.532663,0.000000,inf,2.0,2.0
44299,MAGH,6.760,5.451613,0.000000,inf,4.0,2.0
33181,EFTY,15.020,2.106592,0.000000,inf,8.0,2.0
37942,HCMAU,10.470,1.034585,0.182827,5.385326,34.0,4.0
24513,CEPF,10.355,1.024233,0.351583,2.770994,36.0,5.0
61901,TMDE,0.749,0.776166,0.414673,1.751178,54.0,6.0
61544,TLIH,5.750,0.989673,0.648547,1.448889,39.0,7.0
24940,CEPO,10.755,1.031160,0.795258,1.233763,35.0,8.0
39341,HXHX,0.471,0.409565,0.477354,0.753246,84.0,9.0
35631,FBGL,0.375,0.576923,0.838018,0.628773,67.0,10.0


**Observations**
- Mean `growth_252d` lebih besar dari median → beberapa saham dengan pertumbuhan ekstrem menarik rata-rata ke atas; median lebih mewakili saham "tipikal".
- Karena volatilitas dihitung dari std harga `Close` (bukan return), saham berharga rendah punya volatilitas absolut kecil sehingga Sharpe-nya bisa tinggi meski growth-nya biasa saja — ranking Sharpe berbeda dari ranking growth.

---
## Question 3: [IPO] 'Fixed Months Holding Strategy'

In [12]:
g = stocks_df.groupby('Ticker')['Close']
FUTURE_COLS = []
for m in range(1, 13):
    col = f'future_growth_{m}_m'
    stocks_df[col] = g.shift(-21 * m) / stocks_df['Close']
    FUTURE_COLS.append(col)

# first trading day (entry point) for each ticker
min_dates = stocks_df.groupby('Ticker')['Date'].min().reset_index().rename(columns={'Date': 'min_date'})

first_day = pd.merge(
    min_dates,
    stocks_df[['Ticker', 'Date', 'Close'] + FUTURE_COLS],
    left_on=['Ticker', 'min_date'],
    right_on=['Ticker', 'Date'],
    how='inner',
)
print('Tickers:', len(first_day))
q3_stats = first_day[FUTURE_COLS].describe()
q3_stats

Tickers: 133


,future_growth_1_m,future_growth_2_m,future_growth_3_m,future_growth_4_m,future_growth_5_m,future_growth_6_m,future_growth_7_m,future_growth_8_m,future_growth_9_m,future_growth_10_m,future_growth_11_m,future_growth_12_m
count,132.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,131.000000,130.000000
mean,1.119635,1.294861,1.252014,1.135939,1.084528,1.114789,1.146371,1.004473,1.158480,1.185497,1.068597,0.963118
std,1.254616,1.957999,1.912626,1.558464,1.468733,1.668273,2.278159,2.046907,2.999642,3.422082,2.161964,2.132421
min,0.072941,0.106197,0.090147,0.063927,0.032245,0.022075,0.013229,0.015130,0.014675,0.011947,0.008681,0.004833
25%,0.708835,0.584863,0.442477,0.392759,0.400704,0.308026,0.245548,0.200983,0.174587,0.174333,0.159666,0.156935
50%,0.935351,0.890411,0.827160,0.741678,0.667857,0.726000,0.662000,0.603508,0.580294,0.533333,0.492453,0.529827
75%,1.098038,1.152849,1.149994,1.203576,1.083509,1.056403,1.019297,1.036588,1.031006,1.034765,1.106045,1.041541
max,9.466019,16.066783,16.344464,11.386643,9.495146,9.894552,20.741478,21.370826,27.627298,34.687666,19.595802,22.044619


In [13]:
medians = q3_stats.loc['50%']
best_col = medians.idxmax()
best_month = int(re.search(r'_(\d+)_m', best_col).group(1))
print(pd.DataFrame({'median': medians, 'mean': q3_stats.loc['mean'], 'count': q3_stats.loc['count']}))
print(f"\nANSWER Q3: optimal holding period = {best_month} month(s), max median growth = {medians.max():.4f}")

                      median      mean  count
future_growth_1_m   0.935351  1.119635  132.0
future_growth_2_m   0.890411  1.294861  131.0
future_growth_3_m   0.827160  1.252014  131.0
future_growth_4_m   0.741678  1.135939  131.0
future_growth_5_m   0.667857  1.084528  131.0
future_growth_6_m   0.726000  1.114789  131.0
future_growth_7_m   0.662000  1.146371  131.0
future_growth_8_m   0.603508  1.004473  131.0
future_growth_9_m   0.580294  1.158480  131.0
future_growth_10_m  0.533333  1.185497  131.0
future_growth_11_m  0.492453  1.068597  131.0
future_growth_12_m  0.529827  0.963118  130.0

ANSWER Q3: optimal holding period = 1 month(s), max median growth = 0.9354


**Observations**
- Perhatikan tren median terhadap lama holding: median < 1 berarti saham tipikal rugi dibanding harga penutupan hari pertama → strategi beli-dan-tahan IPO baru umumnya tidak menguntungkan untuk saham tipikal.
- Mean jauh di atas median karena beberapa outlier dengan pertumbuhan sangat tinggi; strategi ini sangat bergantung pada "menemukan pemenang".

---
## Question 4: [Strategy] Simple RSI-Based Trading Strategy

In [14]:
import gdown

file_id = '1grCTCzMZKY5sJRtdbLVCXg8JXA8VPyg-'
if not os.path.exists('data.parquet'):
    gdown.download(f'https://drive.google.com/uc?id={file_id}', 'data.parquet', quiet=False)
df = pd.read_parquet('data.parquet', engine='pyarrow')
print(df.shape)
df[['Date', 'Ticker', 'Close_x', 'rsi', 'growth_future_30d']].head()

(229932, 203)


,Date,Ticker,Close_x,rsi,growth_future_30d
0,1986-03-13,MSFT,0.059598,NaN,0.982144
1,1986-03-14,MSFT,0.061726,NaN,0.922418
2,1986-03-17,MSFT,0.062791,NaN,0.881354
3,1986-03-18,MSFT,0.061194,NaN,0.921744
4,1986-03-19,MSFT,0.060130,NaN,0.964604


In [15]:
RSI_THRESHOLD = 30
df['Date'] = pd.to_datetime(df['Date'], utc=True).dt.tz_localize(None)

selected_df = df[
    (df['rsi'] < RSI_THRESHOLD)
    & (df['Date'] >= '2000-01-01')
    & (df['Date'] <= '2025-06-01')
]

net_income = 1000 * (selected_df['growth_future_30d'] - 1).sum()
print('Number of trades:', len(selected_df))
print(f"Average 30d return: {(selected_df['growth_future_30d'].mean() - 1) * 100:.2f}%")
print(f"Win rate: {(selected_df['growth_future_30d'] > 1).mean() * 100:.2f}%")
print(f"\nANSWER Q4: net income = ${net_income:,.2f} (~{net_income / 1000:.1f} thousand)")

Number of trades: 5206
Average 30d return: 1.26%
Win rate: 55.13%

ANSWER Q4: net income = $65,805.59 (~65.8 thousand)


---
## Q5. [Exploratory, Optional] Predicting a Positive-Return IPO

Ide untuk meningkatkan profitabilitas strategi IPO:

1. **Jangan beli di hari pertama.** Hasil Q3 menunjukkan median return dari harga penutupan hari pertama cenderung negatif. Tunggu fase *post-IPO drift* / berakhirnya **lock-up period** (~90–180 hari) ketika tekanan jual insider sudah lewat, lalu masuk saat harga stabil.
2. **Filter kualitas IPO** sebelum membeli: ukuran deal (Est $ Vol), reputasi underwriter (tier-1 bank), SCOOP Rating, sektor (hindari blank-check/SPAC dan micro-cap), apakah harga dinaikkan di atas range (sinyal permintaan tinggi), dan profitabilitas/pertumbuhan pendapatan perusahaan.
3. **Gunakan sinyal momentum / teknikal** setelah IPO: masuk hanya jika harga di atas SMA20/SMA50 atau breakout dari *IPO base*, dan kombinasikan dengan RSI (Q4) untuk membeli saat oversold.
4. **Kondisi pasar makro**: IPO cenderung berkinerja lebih baik saat pasar bullish (S&P500 di atas SMA200, VIX rendah). Pakai filter regime pasar.
5. **Manajemen risiko**: stop-loss/take-profit, position sizing berbasis volatilitas, dan diversifikasi ke banyak IPO karena return didorong oleh sedikit outlier (mean >> median).
6. **Model ML**: bangun classifier (mis. gradient boosting) dengan fitur di atas untuk memprediksi `is_positive_growth_Nd_future`, lalu hanya trading pada IPO dengan probabilitas prediksi tinggi (precision tinggi, meski trade lebih sedikit).

---
## Summary of Answers

In [16]:
print(f"Q1: {q1.index[0]} -> ${q1.iloc[0]:,.2f} million")
print(f"Q2: median Sharpe (2026-09-11) = {q2['Sharpe'].median():.4f}")
print(f"Q3: {best_month} month(s), median growth = {medians.max():.4f}")
print(f"Q4: net income = ${net_income / 1000:,.1f} thousand")

Q1: Acquisition Corp -> $499.99 million
Q2: median Sharpe (2026-09-11) = 0.0501
Q3: 1 month(s), median growth = 0.9354
Q4: net income = $65.8 thousand
